In [22]:
%pip install nbformat nbclient ipykernel

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: C:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Parte 1.

#### 1. Imports e caminhos

In [23]:
import json
import hashlib
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import nbformat
from nbclient import NotebookClient


ROOT = Path.cwd().resolve()

OUTPUT_DIR = ROOT / "outputs" / "patients"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Alterar quando os datasets limpos forem atualizados
DATA_VERSION = "eicu-clean-v1"

# IDs das células que inicializam cada agente.
# Estes são candidatos baseados nas funções que identificámos;
# confirmar antes de executar.
AGENT_CONFIG = {
    "vital": {
        "notebook": "VitalAgent_imp.ipynb",
        "function": "run_vital_agent",
        "cells": [2, 4, 6, 8, 10, 11, 14],
    },

    "laboratory": {
        "notebook": "LabAgent_imp.ipynb",
        "function": "run_lab_agent",
        "cells": [2, 4, 6, 8, 10, 11, 13],
    },

    "clinical": {
        "notebook": "ClinicalAgent_imp.ipynb",
        "function": "run_clinical_agent",
        "cells": [2, 4, 6, 8, 10, 11, 13],
    },
}

def now():
    return datetime.now(timezone.utc).isoformat()

print("Project:", ROOT)
print("Outputs:", OUTPUT_DIR)

Project: C:\Users\ASUS\Desktop\1º ano m\IACH\projeto\eicu-collaborative-research-database-demo-2.0.1
Outputs: C:\Users\ASUS\Desktop\1º ano m\IACH\projeto\eicu-collaborative-research-database-demo-2.0.1\outputs\patients


#### 2. Inspecionar as células dos agentes

In [24]:
for agent_name, config in AGENT_CONFIG.items():

    path = ROOT / config["notebook"]

    print(f"\n{'=' * 60}")
    print(agent_name.upper(), "-", path.name)
    print("=" * 60)

    notebook = nbformat.read(path, as_version=4)

    for i, cell in enumerate(notebook.cells):
        if cell.cell_type != "code":
            continue

        lines = [
            line.strip()
            for line in cell.source.splitlines()
            if line.strip()
        ]

        preview = " | ".join(lines[:3])

        print(f"Célula {i}: {preview[:180]}")


VITAL - VitalAgent_imp.ipynb
Célula 0: %pip install -q pandas requests pydantic python-dotenv
Célula 2: import os, json, uuid, re, math | from pathlib import Path | from typing import Literal
Célula 4: VITAL_DATA_PATH = locate_csv("vital_agent_data.csv") | vital_df = pd.read_csv(VITAL_DATA_PATH) | if "patientunitstayid" not in vital_df.columns:
Célula 6: class VitalFinding(BaseModel): | model_config = ConfigDict(extra="forbid") | variable: str
Célula 8: def get_vital_data(patientunitstayid: int) -> dict: | """Dados agregados de um internamento; função determinística, sem chamada ao LLM.""" | if patientunitstayid not in vital_df.in
Célula 10: # IAedu comunica por eventos de streaming (SSE). Não enviar informações identificáveis. | def ask_iaedu(message: str) -> str: | """Envia uma mensagem à IAedu e interpreta o streami
Célula 11: def validated_selection(output, patient_id, evaluations): | if output.patient_id != patient_id or output.status == "error": | raise ValueError("ID ou estado 

#### 3. Funções para guardar e validar os JSONs

In [25]:
def patient_folder(patient_id):
    folder = OUTPUT_DIR / str(patient_id)
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def agent_output_path(patient_id, agent_name):
    return patient_folder(patient_id) / f"{agent_name}.json"


def notebook_fingerprint(config):
    """
    Identifica a versão do notebook e das células
    usadas na execução.
    """
    path = ROOT / config["notebook"]

    content = path.read_bytes()

    metadata = json.dumps({
        "cells": config["cells"],
        "function": config["function"],
        "data_version": DATA_VERSION,
    }, sort_keys=True).encode()

    return hashlib.sha256(content + metadata).hexdigest()


def save_json(path, data):
    """
    Guarda JSON localmente.
    Usa ficheiro temporário para evitar resultados
    parcialmente escritos.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_suffix(".tmp")

    temporary.write_text(
        json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False),
        encoding="utf-8"
    )

    temporary.replace(path)


def load_cached_agent(patient_id, agent_name):
    """
    Reutiliza um resultado apenas se:
    - pertencer ao paciente certo;
    - tiver sido produzido pelo agente certo;
    - estiver marcado como success;
    - corresponder à versão esperada.
    """

    path = agent_output_path(patient_id, agent_name)

    if not path.exists():
        return None

    try:
        cached = json.loads(path.read_text(encoding="utf-8"))

        config = AGENT_CONFIG[agent_name]

        valid = (
            isinstance(cached, dict)
            and cached.get("patientunitstayid") == patient_id
            and cached.get("agent") == agent_name
            and cached.get("status") == "success"
            and cached.get("fingerprint")
                == notebook_fingerprint(config)
            and isinstance(cached.get("data"), (dict, list))
            and bool(cached["data"])
        )

        if valid:
            return cached

    except (OSError, ValueError, TypeError):
        pass

    return None

#### 4. Executar diretamente um agente em .ipynb

In [26]:
def execute_notebook_agent(agent_name, patient_id):
    """
    Executa as células selecionadas de um notebook
    e chama a função principal do agente.

    Cada execução ocorre num kernel Jupyter isolado.
    """

    config = AGENT_CONFIG[agent_name]

    if not config["cells"]:
        raise ValueError(
            f"Cell configuration missing for {agent_name}"
        )

    path = ROOT / config["notebook"]

    original = nbformat.read(path, as_version=4)

    selected_cells = []

    for index in config["cells"]:

        if not isinstance(index, int):
            raise ValueError("Cell indices must be integers")

        if index < 0 or index >= len(original.cells):
            raise IndexError(
                f"Invalid cell index {index} in {path.name}"
            )

        cell = original.cells[index]

        if cell.cell_type != "code":
            raise ValueError(
                f"Cell {index} is not a code cell"
            )

        selected_cells.append(
            nbformat.v4.new_code_cell(cell.source)
        )

    with tempfile.TemporaryDirectory() as temp_dir:

        result_path = Path(temp_dir) / "agent_result.json"

        # Esta célula adicional é executada no fim.
        # Chama a função principal e guarda o resultado.
        call_code = f"""
import json
import math
from pathlib import Path

def _serialize(value):

    if hasattr(value, "model_dump"):
        return _serialize(value.model_dump(mode="json"))

    if hasattr(value, "to_dict"):
        return _serialize(value.to_dict())

    if isinstance(value, dict):
        return {{
            str(k): _serialize(v)
            for k, v in value.items()
        }}

    if isinstance(value, (list, tuple)):
        return [_serialize(v) for v in value]

    if hasattr(value, "item"):
        return _serialize(value.item())

    if isinstance(value, float) and not math.isfinite(value):
        return None

    if isinstance(value, (str, int, float, bool)) or value is None:
        return value

    return str(value)

_patient_id = {patient_id!r}

_agent_result = {config["function"]}(_patient_id)

Path({str(result_path)!r}).write_text(
    json.dumps(
        _serialize(_agent_result),
        ensure_ascii=False,
        allow_nan=False
    ),
    encoding="utf-8"
)
"""

        selected_cells.append(
            nbformat.v4.new_code_cell(call_code)
        )

        notebook = nbformat.v4.new_notebook(
            cells=selected_cells
        )

        client = NotebookClient(
            notebook,
            timeout=300,
            kernel_name="python3",
            resources={
                "metadata": {
                    "path": str(ROOT)
                }
            },
            allow_errors=False
        )

        client.execute()

        if not result_path.exists():
            raise RuntimeError(
                f"{agent_name} did not produce a result"
            )

        return json.loads(
            result_path.read_text(encoding="utf-8")
        )

#### 5. Executar ou reutilizar o Data Agent

In [27]:
def get_or_run_agent(agent_name, patient_id):

    config = AGENT_CONFIG[agent_name]

    # 1. Verificar resultado existente
    cached = load_cached_agent(patient_id, agent_name)

    if cached is not None:
        print(f"[CACHE] {agent_name}")
        return cached

    # 2. Executar agente
    print(f"[RUN] {agent_name}")

    try:
        raw = execute_notebook_agent(
            agent_name,
            patient_id
        )

        if not isinstance(raw, (dict, list)):
            raise ValueError(
                "Agent returned a non-structured result"
            )

        # Respeitar o estado devolvido pelo agente
        if isinstance(raw, dict) and "status" in raw:

            status = raw["status"]

            if status not in (
                "success",
                "insufficient_data",
                "error"
            ):
                raise ValueError(
                    f"Invalid agent status: {status}"
                )

            data = raw.get("data")

            if data is None:
                data = {
                    k: v for k, v in raw.items()
                    if k not in {"status", "error"}
                }

            error = raw.get("error")

        else:
            status = "success"
            data = raw
            error = None

        if status == "success" and not data:
            status = "insufficient_data"

    except Exception as exc:

        status = "error"
        data = None
        error = f"{type(exc).__name__}: {exc}"

        print(f"[ERROR] {agent_name}: {error}")

    # 3. Construir resultado padronizado
    result = {
        "patientunitstayid": patient_id,
        "agent": agent_name,
        "status": status,
        "data": data,
        "error": error,
        "fingerprint": notebook_fingerprint(config),
        "created_at": now()
    }

    # 4. Guardar resultado
    save_json(
        agent_output_path(patient_id, agent_name),
        result
    )

    return result

#### 6. Orchestrator principal

In [28]:
def run_orchestrator(
    patient_id,
    severity_runner=None,
    clean_data_loader=None
):
    """
    Coordena os três Data Agents e, opcionalmente,
    o Severity Agent.

    severity_runner:
        Função que recebe o severity_input.

    clean_data_loader:
        Função que recebe patient_id e devolve
        os dados limpos desse paciente.
    """

    if (
        isinstance(patient_id, bool)
        or not isinstance(patient_id, int)
        or patient_id <= 0
    ):
        raise ValueError(
            "patient_id must be a positive integer"
        )

    print(f"\n=== IACH | Patient {patient_id} ===")

    # 1. Executar ou reutilizar os Data Agents
    agents = {}

    for agent_name in AGENT_CONFIG:
        agents[agent_name] = get_or_run_agent(
            agent_name,
            patient_id
        )

    # 2. Verificar se todos tiveram sucesso
    all_success = all(
        result["status"] == "success"
        for result in agents.values()
    )

    statuses = [result["status"] for result in agents.values()]

    if all(s == "success" for s in statuses):
        overall_status = "success"
    elif all(s == "error" for s in statuses):
        overall_status = "error"
    else:
        overall_status = "partial"

    state = {
        "patientunitstayid": patient_id,
        "status": overall_status,
        "agents": agents,
        "ready_for_severity": overall_status == "success",
        "created_at": now()
    }

    # 3. Guardar estado unificado
    save_json(
        patient_folder(patient_id) / "patient_state.json",
        state
    )

    if not all_success:
        print("[STOP] Data agents incomplete")
        return state

    print("[OK] All Data Agents completed")

    # 4. Caso não tenhamos ainda o Severity Agent,
    # terminar após preparar o estado.
    if severity_runner is None:
        print("[READY] Data available for Severity Agent")
        return state

    if clean_data_loader is None:
        raise ValueError(
            "clean_data_loader is required for severity"
        )

    # 5. Obter as características limpas deste paciente
    features = clean_data_loader(patient_id)

    if not isinstance(features, dict) or not features:
        raise ValueError(
            "Missing or invalid cleaned patient features"
        )

    # 6. Construir input do Severity Agent
    severity_input = {
        "patientunitstayid": patient_id,
        "features": features,
        "agent_results": {
            name: result["data"]
            for name, result in agents.items()
        }
    }

    # 7. Executar Severity Agent
    print("[RUN] Severity Agent")

    severity_result = severity_runner(severity_input)

    if not isinstance(severity_result, dict):
        raise ValueError(
            "Severity Agent must return a dictionary"
        )

    # 8. Guardar previsão
    save_json(
        patient_folder(patient_id) / "severity.json",
        {
            "patientunitstayid": patient_id,
            "result": severity_result,
            "created_at": now()
        }
    )

    state["severity"] = severity_result

    print("[OK] Severity assessment completed")

    return state

#### 8. Execução com os agentes reais

In [29]:
patient_id = 148611  # Substituir por um ID real do eICU

state = run_orchestrator(patient_id)

print(json.dumps(
    {
        "patientunitstayid": state["patientunitstayid"],
        "status": state["status"],
        "ready_for_severity": state["ready_for_severity"],
        "agent_statuses": {
            name: result["status"]
            for name, result in state["agents"].items()
        }
    },
    indent=2
))


=== IACH | Patient 148611 ===
[CACHE] vital
[CACHE] laboratory
[CACHE] clinical
[OK] All Data Agents completed
[READY] Data available for Severity Agent
{
  "patientunitstayid": 148611,
  "status": "success",
  "ready_for_severity": true,
  "agent_statuses": {
    "vital": "success",
    "laboratory": "success",
    "clinical": "success"
  }
}
